In [22]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import fastf1 as f1
from sklearn.linear_model import LinearRegression, RANSACRegressor

# Import local class (classes/F1Analyzer.py)
classes_dir = next((p / 'classes' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'classes' / 'F1Analyzer.py').exists()), None)
if classes_dir is None:
    raise FileNotFoundError('Impossible de trouver classes/F1Analyzer.py depuis le dossier courant')
sys.path.append(str(classes_dir.resolve()))
from F1Analyzer import F1Analyzer

f1.set_log_level('WARNING')
f1.Cache.enable_cache('cache_strategy')

# Fallback si le dictionnaire de precision n'est pas encore charge.
CIRCUIT_STATS = {
    'Bahrain Grand Prix': {'abrasivity': 5, 'lateral_energy': 3},
    'Saudi Arabian Grand Prix': {'abrasivity': 2, 'lateral_energy': 4},
    'Australian Grand Prix': {'abrasivity': 3, 'lateral_energy': 3},
    'Japanese Grand Prix': {'abrasivity': 5, 'lateral_energy': 5},
    'Chinese Grand Prix': {'abrasivity': 3, 'lateral_energy': 4},
    'Miami Grand Prix': {'abrasivity': 3, 'lateral_energy': 3},
    'Emilia Romagna Grand Prix': {'abrasivity': 3, 'lateral_energy': 3},
    'Monaco Grand Prix': {'abrasivity': 1, 'lateral_energy': 1},
    'Canadian Grand Prix': {'abrasivity': 2, 'lateral_energy': 2},
    'Spanish Grand Prix': {'abrasivity': 4, 'lateral_energy': 5},
    'Austrian Grand Prix': {'abrasivity': 2, 'lateral_energy': 3},
    'British Grand Prix': {'abrasivity': 3, 'lateral_energy': 5},
    'Hungarian Grand Prix': {'abrasivity': 2, 'lateral_energy': 3},
    'Belgian Grand Prix': {'abrasivity': 4, 'lateral_energy': 5},
    'Dutch Grand Prix': {'abrasivity': 3, 'lateral_energy': 5},
    'Italian Grand Prix': {'abrasivity': 2, 'lateral_energy': 3},
    'Azerbaijan Grand Prix': {'abrasivity': 2, 'lateral_energy': 2},
    'Singapore Grand Prix': {'abrasivity': 4, 'lateral_energy': 2},
    'United States Grand Prix': {'abrasivity': 4, 'lateral_energy': 4},
    'Mexico City Grand Prix': {'abrasivity': 2, 'lateral_energy': 2},
    'Sao Paulo Grand Prix': {'abrasivity': 3, 'lateral_energy': 3},
    'Las Vegas Grand Prix': {'abrasivity': 2, 'lateral_energy': 2},
    'Qatar Grand Prix': {'abrasivity': 4, 'lateral_energy': 5},
    'Abu Dhabi Grand Prix': {'abrasivity': 3, 'lateral_energy': 3},
}

COMPOUND_MAP = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2}


def _cache_event_folder_to_name(folder_name):
    part = folder_name.split('_', 1)[1]
    return part.replace('_', ' ')


def get_cached_race_events(cache_root='cache_strategy', year=2024):
    root = Path(cache_root) / str(year)
    if not root.exists():
        return []

    events = []
    for event_dir in sorted([d for d in root.iterdir() if d.is_dir()]):
        has_race = any(child.is_dir() and child.name.endswith('_Race') for child in event_dir.iterdir())
        if has_race:
            events.append(_cache_event_folder_to_name(event_dir.name))

    events = [e for e in events if e != 'United States Grand Prix']
    return events


def get_circuit_features(event_name):
    alias_to_precision = {
        'Sao Paulo Grand Prix': 'São Paulo Grand Prix'
    }
    normalized_event = alias_to_precision.get(event_name, event_name)

    stats_precision = globals().get('f1_circuit_precision_stats', None)
    if isinstance(stats_precision, dict) and normalized_event in stats_precision:
        info = stats_precision[normalized_event]
        return float(info.get('abrasivity', np.nan)), float(info.get('lat_energy', np.nan))

    fallback = CIRCUIT_STATS.get(event_name, {})
    return fallback.get('abrasivity', np.nan), fallback.get('lateral_energy', np.nan)


# 1. Extraire TrackTemp depuis weather_data proprement
def get_tracktemp_map(session):
    try:
        wd = session.weather_data.copy()
        wd['Time'] = pd.to_timedelta(wd['Time'], errors='coerce')
        wd['TrackTemp'] = pd.to_numeric(wd['TrackTemp'], errors='coerce')
        wd = wd.dropna(subset=['Time', 'TrackTemp']).sort_values('Time')
        return wd[['Time', 'TrackTemp']].reset_index(drop=True)
    except Exception:
        return None


# 2. Merger sur LapStartTime (pas Time !)
def assign_tracktemp_to_laps(laps, weather_map):
    if weather_map is None or weather_map.empty:
        laps['TrackTemp'] = np.nan
        return laps

    # FastF1 utilise LapStartTime pour situer chaque tour
    time_col = None
    for col in ['LapStartTime', 'Time']:
        if col in laps.columns:
            time_col = col
            break

    if time_col is None:
        laps['TrackTemp'] = np.nan
        return laps

    laps_sorted = laps.copy()
    laps_sorted['_time_td'] = pd.to_timedelta(
        laps_sorted[time_col], errors='coerce'
    )
    laps_sorted = laps_sorted.sort_values('_time_td')

    merged = pd.merge_asof(
        laps_sorted[['_time_td']],
        weather_map.rename(columns={'Time': '_time_td'}),
        on='_time_td',
        direction='nearest'
    )

    laps.loc[laps_sorted.index, 'TrackTemp'] = merged['TrackTemp'].to_numpy()
    laps = laps.drop(columns=['_time_td'], errors='ignore')
    return laps


def build_event_driver_laps(year, event_name, session_type='R'):
    analyzer = F1Analyzer(year, event_name, session_type, use_fuel_logic=True)
    session = analyzer.session

    all_rows = []

    if hasattr(session, 'total_laps') and session.total_laps is not None:
        race_max_laps = int(session.total_laps)
    else:
        race_max_laps = int(pd.to_numeric(session.laps['LapNumber'], errors='coerce').max())

    abrasivity, lateral_energy = get_circuit_features(event_name)

    weather_map = get_tracktemp_map(session)

    for drv in session.drivers:
        laps = analyzer.get_clean_laps(drv)
        if laps.empty:
            continue

        laps['Compound'] = laps['Compound'].astype(str).str.upper()
        laps = laps[laps['Compound'].isin(COMPOUND_MAP.keys())].copy()
        if laps.empty:
            continue

        laps['LapNumber'] = pd.to_numeric(laps['LapNumber'], errors='coerce')
        laps = laps[laps['LapNumber'].notna()].copy()
        if laps.empty:
            continue

        laps['FuelLoad'] = race_max_laps - laps['LapNumber']

        if 'TyreLife' not in laps.columns:
            laps['TyreLife'] = np.nan
        laps['TyreLife'] = pd.to_numeric(laps['TyreLife'], errors='coerce')
        missing_tyre = laps['TyreLife'].isna()
        if missing_tyre.any():
            laps.loc[missing_tyre, 'TyreLife'] = (
                laps.loc[missing_tyre]
                .groupby(['Stint'], dropna=False)
                .cumcount() + 1
            )

        # Build CorrectedLapTime_Global using the class pipeline.
        laps = analyzer.add_drs_correction_to_laps(laps)
        laps = analyzer.add_dirty_air_correction_to_laps(laps)
        laps = analyzer.add_track_evolution_correction_to_laps(laps)
        if 'CorrectedLapTime_Global' not in laps.columns:
            continue

        laps['CorrectedLapTime_Global'] = pd.to_numeric(laps['CorrectedLapTime_Global'], errors='coerce')
        laps = laps[laps['CorrectedLapTime_Global'].notna()].copy()
        if laps.empty:
            continue

        laps = assign_tracktemp_to_laps(laps, weather_map)
        if 'TrackTemp_C' in laps.columns:
            laps['TrackTemp'] = pd.to_numeric(laps['TrackTemp_C'], errors='coerce').fillna(laps['TrackTemp'])
        elif 'TrackTemp' in laps.columns:
            laps['TrackTemp'] = pd.to_numeric(laps['TrackTemp'], errors='coerce')

        laps['is_inlier_ransac'] = True
        for _, grp in laps.groupby(['Stint', 'Compound'], dropna=False):
            if len(grp) < 5:
                continue

            x = grp[['TyreLife']].to_numpy(dtype=float)
            y = grp['CorrectedLapTime_Global'].to_numpy(dtype=float)

            valid = np.isfinite(x).ravel() & np.isfinite(y)
            if valid.sum() < 5:
                continue

            x_valid = x[valid]
            y_valid = y[valid]
            idx_valid = grp.index[valid]

            try:
                ransac = RANSACRegressor(
                    estimator=LinearRegression(),
                    min_samples=max(3, int(0.5 * len(x_valid))),
                    residual_threshold=0.8,
                    random_state=42
                )
                ransac.fit(x_valid, y_valid)
                laps.loc[idx_valid, 'is_inlier_ransac'] = ransac.inlier_mask_
            except Exception:
                laps.loc[idx_valid, 'is_inlier_ransac'] = True

        laps = laps[laps['is_inlier_ransac']].copy()
        if laps.empty:
            continue

        # Delta calcule par STINT (et non global pilote).
        laps['BestCorrectedByStint'] = laps.groupby('Stint', dropna=False)['CorrectedLapTime_Global'].transform('min')
        laps['DeltaToBest'] = laps['CorrectedLapTime_Global'] - laps['BestCorrectedByStint']

        laps['Year'] = year
        laps['Event'] = event_name
        laps['Driver'] = drv
        laps['Abrasivity'] = abrasivity
        laps['LateralEnergy'] = lateral_energy
        laps['CompoundEncoded'] = laps['Compound'].map(COMPOUND_MAP)

        cols = [
            'Year', 'Event', 'Driver', 'Compound', 'CompoundEncoded', 'TyreLife',
            'TrackTemp', 'FuelLoad', 'Abrasivity', 'LateralEnergy', 'DeltaToBest',
            'LapNumber', 'Stint', 'CorrectedLapTime_Global', 'BestCorrectedByStint'
        ]
        all_rows.append(laps[cols])

    if not all_rows:
        return pd.DataFrame()

    return pd.concat(all_rows, axis=0, ignore_index=True)


def build_master_dataset_from_cached(year=2024, target_rows=20000):
    events = get_cached_race_events(year=year)

    datasets = []
    for event in events:
        try:
            event_df = build_event_driver_laps(year=year, event_name=event, session_type='R')
            if not event_df.empty:
                datasets.append(event_df)
                print(f'[OK] {event}: {len(event_df)} lignes')
            else:
                print(f'[VIDE] {event}: 0 ligne')
        except Exception as exc:
            print(f'[ERREUR] {event}: {exc}')

    if not datasets:
        return pd.DataFrame(), events

    master = pd.concat(datasets, axis=0, ignore_index=True)

    if 'TrackTemp' in master.columns:
        master['TrackTemp'] = pd.to_numeric(master['TrackTemp'], errors='coerce')

    master = master.replace([np.inf, -np.inf], np.nan).dropna(subset=[
        'CompoundEncoded', 'TyreLife', 'TrackTemp', 'FuelLoad',
        'Abrasivity', 'LateralEnergy', 'DeltaToBest', 'CorrectedLapTime_Global'
    ])

    if len(master) > target_rows:
        master = master.sample(n=target_rows, random_state=42).sort_values(['Year', 'Event', 'Driver', 'LapNumber'])
    else:
        master = master.sort_values(['Year', 'Event', 'Driver', 'LapNumber'])

    master = master.reset_index(drop=True)
    return master, events

In [23]:
f1_circuit_precision_stats = {
    "Bahrain Grand Prix": {"lat_energy": 3.2, "abrasivity": 4.8, "type": "Traction"},
    "Saudi Arabian Grand Prix": {"lat_energy": 3.9, "abrasivity": 2.1, "type": "High-Speed Street"},
    "Australian Grand Prix": {"lat_energy": 3.1, "abrasivity": 2.8, "type": "Semi-Permanent"},
    "Japanese Grand Prix": {"lat_energy": 5.0, "abrasivity": 4.6, "type": "High-Energy"},
    "Chinese Grand Prix": {"lat_energy": 4.1, "abrasivity": 3.2, "type": "Front-Limited"},
    "Miami Grand Prix": {"lat_energy": 2.9, "abrasivity": 2.7, "type": "Street"},
    "Emilia Romagna Grand Prix": {"lat_energy": 3.4, "abrasivity": 3.1, "type": "Technical"},
    "Monaco Grand Prix": {"lat_energy": 1.2, "abrasivity": 1.1, "type": "Low-Speed Street"},
    "Canadian Grand Prix": {"lat_energy": 1.8, "abrasivity": 2.3, "type": "Stop-and-Go"},
    "Spanish Grand Prix": {"lat_energy": 4.7, "abrasivity": 4.2, "type": "Aero-Reference"},
    "Austrian Grand Prix": {"lat_energy": 3.1, "abrasivity": 2.4, "type": "Short/Fast"},
    "British Grand Prix": {"lat_energy": 4.9, "abrasivity": 3.3, "type": "High-Energy"},
    "Hungarian Grand Prix": {"lat_energy": 2.8, "abrasivity": 2.2, "type": "Twisty"},
    "Belgian Grand Prix": {"lat_energy": 4.6, "abrasivity": 3.8, "type": "Power/Energy"},
    "Dutch Grand Prix": {"lat_energy": 4.4, "abrasivity": 3.1, "type": "Banking"},
    "Italian Grand Prix": {"lat_energy": 2.7, "abrasivity": 2.2, "type": "Ultra-High Speed"},
    "Azerbaijan Grand Prix": {"lat_energy": 2.1, "abrasivity": 2.4, "type": "Street/Straight"},
    "Singapore Grand Prix": {"lat_energy": 1.9, "abrasivity": 4.1, "type": "Street/Bumpy"},
    "United States Grand Prix": {"lat_energy": 4.2, "abrasivity": 4.3, "type": "Technical/Bumpy"},
    "Mexico City Grand Prix": {"lat_energy": 2.3, "abrasivity": 2.1, "type": "Altitude"},
    "São Paulo Grand Prix": {"lat_energy": 3.2, "abrasivity": 3.4, "type": "Technical"},
    "Las Vegas Grand Prix": {"lat_energy": 1.8, "abrasivity": 1.9, "type": "Cold Street"},
    "Qatar Grand Prix": {"lat_energy": 5.0, "abrasivity": 3.9, "type": "Extreme Lateral"},
    "Abu Dhabi Grand Prix": {"lat_energy": 3.1, "abrasivity": 3.1, "type": "Standard"}
}

In [24]:
# Controle rapide de distribution par GP et pilote
if master_dataset.empty:
    print('Dataset vide')
else:
    display(master_dataset.groupby('Event').size().rename('rows_per_event').sort_values(ascending=False))
    display(master_dataset.groupby(['Event', 'Driver']).size().rename('rows_per_driver_event').head(30))

Dataset vide


In [ ]:
year_to_build = 2023

# Build dataset directly from already downloaded cache.
master_dataset, used_events = build_master_dataset_from_cached(year=year_to_build, target_rows=20000)

print(f"\n=== Resume extraction {year_to_build} ===")
print('Nombre de GP utilises (cache, hors United States GP):', len(used_events))
print('Nombre de lignes dataset:', len(master_dataset))
print('Colonnes:', master_dataset.columns.tolist())

display(master_dataset.head(20))

output_path = Path('classes') / f'master_dataset_partie2_{year_to_build}.csv'
master_dataset.to_csv(output_path, index=False)
print('CSV exporte:', output_path)


=== Caching race sessions for 2023 ===
[CACHE OK] 2023 - Bahrain Grand Prix


core        WARNING 	Driver 11 completed the race distance 00:00.035000 before the recorded end of the session.
_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!
_api        WARNING 	Driver 243: Position data is incomplete!


[CACHE OK] 2023 - Saudi Arabian Grand Prix


_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!
_api        WARNING 	Driver 243: Position data is incomplete!


[CACHE OK] 2023 - Australian Grand Prix


...

core        WARNING 	Driver 1 completed the race distance 00:00.001000 before the recorded end of the session.
_api        WARNING 	Driver 241: Position data is incomplete!
_api        WARNING 	Driver 242: Position data is incomplete!
_api        WARNING 	Driver 243: Position data is incomplete!


[CACHE OK] 2023 - Las Vegas Grand Prix
[CACHE OK] 2023 - Abu Dhabi Grand Prix

=== Caching race sessions for 2024 ===
[CACHE OK] 2024 - Bahrain Grand Prix
[CACHE OK] 2024 - Saudi Arabian Grand Prix
[CACHE OK] 2024 - Australian Grand Prix
[CACHE OK] 2024 - Japanese Grand Prix


core        WARNING 	Driver 1 completed the race distance 00:08.313000 before the recorded end of the session.


[CACHE OK] 2024 - Chinese Grand Prix


KeyboardInterrupt: 

In [ ]:
output_path_fallback = Path('classes') / 'master_dataset_partie2_2023_stint.csv'
master_dataset.to_csv(output_path_fallback, index=False)
print('CSV exporte (fallback):', output_path_fallback)
master_dataset[master_dataset['Event'] == 'Bahrain Grand Prix'].head(30)

CSV exporte (fallback): classes\master_dataset_partie2_2024_stint.csv


,Year,Event,Driver,Compound,CompoundEncoded,TyreLife,TrackTemp,FuelLoad,Abrasivity,LateralEnergy,DeltaToBest,LapNumber,Stint,CorrectedLapTime_Global,BestCorrectedByStint
3633,2024,Bahrain Grand Prix,1,SOFT,0,5.0,23.8,55.0,4.8,3.2,0.000,2.0,1.0,96.330,96.33
3634,2024,Bahrain Grand Prix,1,SOFT,0,6.0,23.8,54.0,4.8,3.2,0.474,3.0,1.0,96.804,96.33
3635,2024,Bahrain Grand Prix,1,SOFT,0,7.0,23.7,53.0,4.8,3.2,0.385,4.0,1.0,96.715,96.33
3636,2024,Bahrain Grand Prix,1,SOFT,0,8.0,23.6,52.0,4.8,3.2,0.928,5.0,1.0,97.258,96.33
3637,2024,Bahrain Grand Prix,1,SOFT,0,9.0,23.5,51.0,4.8,3.2,0.864,6.0,1.0,97.194,96.33
3638,2024,Bahrain Grand Prix,1,SOFT,0,10.0,23.5,50.0,4.8,3.2,0.827,7.0,1.0,97.157,96.33
3639,2024,Bahrain Grand Prix,1,SOFT,0,11.0,23.7,49.0,4.8,3.2,0.830,8.0,1.0,97.160,96.33
3640,2024,Bahrain Grand Prix,1,SOFT,0,12.0,23.5,48.0,4.8,3.2,1.052,9.0,1.0,97.382,96.33
3641,2024,Bahrain Grand Prix,1,SOFT,0,15.0,23.5,45.0,4.8,3.2,0.919,12.0,1.0,97.249,96.33
3642,2024,Bahrain Grand Prix,1,SOFT,0,16.0,23.6,44.0,4.8,3.2,0.921,13.0,1.0,97.251,96.33
